# Mera Music AI — One Click Setup
Run the single code cell below after connecting a Colab GPU runtime. Keep the runtime open while generating songs.


In [ ]:
import os, sys, subprocess, time, json, re, pathlib, urllib.request, shutil

# 1) Set dtype BEFORE ACE-Step starts (prevents the T4 float16 NaN/Inf issue)
os.environ['ACESTEP_DTYPE'] = 'float32'
os.environ['MPLBACKEND'] = 'Agg'

# 2) Get ACE-Step source and install its pinned dependencies
workdir = '/content/ACE-Step-1.5'
if not os.path.isdir(workdir):
    subprocess.run(['git','clone','https://github.com/ace-step/ACE-Step-1.5.git',workdir], check=True)
subprocess.run(['bash','-lc','uv sync'], cwd=workdir, check=True)

# 2b) Enable browser CORS for the GitHub Pages frontend before starting FastAPI
api_file = pathlib.Path(workdir) / 'acestep' / 'api_server.py'
api_source = api_file.read_text(encoding='utf-8')
cors_import = 'from fastapi.middleware.cors import CORSMiddleware'
if cors_import not in api_source:
    api_source = api_source.replace(
        'from fastapi import FastAPI\n',
        'from fastapi import FastAPI\nfrom fastapi.middleware.cors import CORSMiddleware\n',
        1
    )
cors_line = 'app.add_middleware(CORSMiddleware, allow_origins=["https://pa12345pp-cmyk.github.io"], allow_methods=["*"], allow_headers=["*"], allow_credentials=False)'
if cors_line not in api_source:
    app_marker = '\napp = create_app()\n'
    if app_marker not in api_source:
        raise RuntimeError('Could not safely locate app = create_app() in api_server.py; CORS patch not applied.')
    api_source = api_source.replace(app_marker, app_marker + cors_line + '\n', 1)
if cors_import not in api_source or cors_line not in api_source:
    raise RuntimeError('CORS verification failed; refusing to start server.')
api_file.write_text(api_source, encoding='utf-8')
print('CORS enabled for https://pa12345pp-cmyk.github.io')

# 3) Start API in background so this cell can finish and the notebook remains usable
log_path = '/content/acestep-api.log'
log_file = open(log_path,'w')
env = os.environ.copy()
api = subprocess.Popen(['uv','run','acestep-api'], cwd=workdir, env=env, stdout=log_file, stderr=subprocess.STDOUT, start_new_session=True)
base = 'http://127.0.0.1:8001'
import requests
for n in range(180):
    try:
        h = requests.get(base+'/health', timeout=3)
        if h.ok: break
    except Exception: pass
    if api.poll() is not None:
        raise RuntimeError('ACE-Step server exited. Last log lines:\n' + '\n'.join(pathlib.Path(log_path).read_text(errors='ignore').splitlines()[-35:]))
    time.sleep(5)
else:
    raise TimeoutError('ACE-Step API did not start. Check /content/acestep-api.log')
print('API server is responding:', h.status_code)

# 4) Initialize Turbo model without the optional LM; generation uses thinking=False
try:
    init = requests.post(base+'/v1/init', json={'model':'acestep-v15-turbo','init_llm':False}, timeout=900)
    print('Model initialization HTTP:', init.status_code)
    print(init.text[:3000])
    init.raise_for_status()
except requests.exceptions.ReadTimeout:
    print('Initialization request timed out; checking health. Do not start another init yet.')
    print(requests.get(base+'/health',timeout=15).text)

# 5) Install cloudflared and create a temporary HTTPS tunnel
cloudflared = '/content/cloudflared'
if not os.path.isfile(cloudflared):
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', cloudflared)
    os.chmod(cloudflared,0o755)
tunnel_log = open('/content/cloudflared.log','w')
tunnel = subprocess.Popen([cloudflared,'tunnel','--url',base,'--no-autoupdate'], stdout=tunnel_log, stderr=subprocess.STDOUT, start_new_session=True)
public_url = None
for n in range(90):
    try:
        txt = pathlib.Path('/content/cloudflared.log').read_text(errors='ignore')
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',txt)
        if m: public_url=m.group(0); break
    except Exception: pass
    if tunnel.poll() is not None: break
    time.sleep(2)
if not public_url:
    print('Tunnel did not return a URL. Check /content/cloudflared.log')
else:
    print('\nMERA MUSIC AI API URL:',public_url)
    try:
        hh=requests.get(public_url+'/health',timeout=30)
        print('Public API health:',hh.status_code,hh.text[:1200])
    except Exception as e: print('Public health check pending:',repr(e))
print('\nSetup cell finished. Keep this Colab runtime open while using the website.')
